# Day 4

## Tokenizing with code

In [5]:
# Cell 1: Imports and setup
import os
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if not groq_api_key:
    print("No Groq API key was found - please add GROQ_API_KEY to your .env file")
elif not groq_api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start with 'gsk_' - please check you're using the right Groq API key")
else:
    print("Groq API key found and looks good so far!")

# Configure Groq client
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

Groq API key found and looks good so far!


In [6]:
encoding = tiktoken.encoding_for_model("gpt-4")  
tokens = encoding.encode("Hi my name is Jay and I like lasagna")

In [7]:
print(f"Tokens: {tokens}")

Tokens: [13347, 856, 836, 374, 19455, 323, 358, 1093, 5252, 56057]


In [8]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

13347 = Hi
856 =  my
836 =  name
374 =  is
19455 =  Jay
323 =  and
358 =  I
1093 =  like
5252 =  las
56057 = agna


In [10]:
encoding.decode([323])

' and'

### A message to OpenAI is a list of dicts

In [12]:

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Hello, are you working?"}]
)
print(response.choices[0].message.content)

# Check token usage
if hasattr(response, 'usage'):
    print(f"\nToken usage:")
    print(f"  Prompt tokens: {response.usage.prompt_tokens}")
    print(f"  Completion tokens: {response.usage.completion_tokens}")
    print(f"  Total tokens: {response.usage.total_tokens}")

Hello! I'm here and ready to help. What can I do for you today?

Token usage:
  Prompt tokens: 77
  Completion tokens: 55
  Total tokens: 132


In [13]:
# Cell 5: The Illusion of Memory - First attempt (stateless)
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Jay!"}
]

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b", 
    messages=messages
)
print(f"Assistant: {response.choices[0].message.content}")

Assistant: Hi Jay! 👋 Nice to meet you. How can I help you today?


### OK let's now ask a follow-up question

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [15]:
# Cell 6: The Illusion of Memory - Follow-up without context
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
]

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b", 
    messages=messages
)
print(f"Assistant: {response.choices[0].message.content}")
print("\n❌ It forgot because each call is stateless!")

Assistant: I’m not sure—could you tell me your name?

❌ It forgot because each call is stateless!


### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [16]:

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Jay!"},
    {"role": "assistant", "content": "Hi Jay! 👋 Nice to meet you. How can I help you today?"},
    {"role": "user", "content": "What's my name?"}
]

response = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b", 
    messages=messages
)
print(f"Assistant: {response.choices[0].message.content}")
print("\n✅ Now it remembers because we passed the whole conversation!")

# Show token usage for this conversation
if hasattr(response, 'usage'):
    print(f"\nToken usage for this request:")
    print(f"  Prompt tokens: {response.usage.prompt_tokens}")
    print(f"  Completion tokens: {response.usage.completion_tokens}")
    print(f"  Total tokens: {response.usage.total_tokens}")

Assistant: Your name is Jay!

✅ Now it remembers because we passed the whole conversation!

Token usage for this request:
  Prompt tokens: 115
  Completion tokens: 58
  Total tokens: 173


In [18]:
def chat_with_memory():
    conversation = [
        {"role": "system", "content": "You are a helpful assistant"}
    ]
    
    print("=" * 60)
    print("CHAT WITH MEMORY DEMONSTRATION")
    print("=" * 60)
    
    # First message
    user_msg = "Hi! I'm Jay!"
    print(f"\nUser: {user_msg}")
    conversation.append({"role": "user", "content": user_msg})
    
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=conversation
    )
    
    assistant_reply = response.choices[0].message.content
    print(f"Assistant: {assistant_reply}")
    conversation.append({"role": "assistant", "content": assistant_reply})
    
    print(f"Tokens used so far: {response.usage.total_tokens if hasattr(response, 'usage') else 'N/A'}")
    
    print("\n" + "-" * 40)
    
    # Second message - asking about name
    user_msg = "What's my name?"
    print(f"\nUser: {user_msg}")
    conversation.append({"role": "user", "content": user_msg})
    
    # Estimate tokens before sending
    if hasattr(response, 'usage'):
        print(f"Conversation history length: {len(conversation)} messages")
    
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=conversation
    )
    
    assistant_reply = response.choices[0].message.content
    print(f"Assistant: {assistant_reply}")
    
    if hasattr(response, 'usage'):
        print(f"\nFinal token usage:")
        print(f"  Total conversation tokens: {response.usage.total_tokens}")
        print(f"  Prompt tokens (history + new): {response.usage.prompt_tokens}")
        print(f"  Completion tokens: {response.usage.completion_tokens}")

chat_with_memory()

CHAT WITH MEMORY DEMONSTRATION

User: Hi! I'm Jay!
Assistant: Hi Jay! 👋 How can I help you today?
Tokens used so far: 129

----------------------------------------

User: What's my name?
Conversation history length: 4 messages
Assistant: Your name is Jay.

Final token usage:
  Total conversation tokens: 152
  Prompt tokens (history + new): 110
  Completion tokens: 42


In [21]:
# Cell 9: Show token growth over conversation with visible responses
def demonstrate_token_growth():
    conversation = [{"role": "system", "content": "You are a helpful assistant"}]
    total_tokens = 0
    
    messages_to_send = [
        "Hi, I'm Jay!",
        "I live in surat",
        "I work as an computer engineer working in aiml",
        "What do you know about me?"
    ]
    
    print("=" * 70)
    print("TOKEN GROWTH DEMONSTRATION")
    print("=" * 70)
    
    for i, msg in enumerate(messages_to_send, 1):
        print(f"\n[{i}] User: {msg}")
        conversation.append({"role": "user", "content": msg})
        
        # Make API call
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=conversation
        )
        
        # Get and print assistant response
        assistant_response = response.choices[0].message.content
        print(f"[{i}] Assistant: {assistant_response}")
        
        # Add assistant response to conversation for next round
        conversation.append({"role": "assistant", "content": assistant_response})
        
        # Print token usage
        if hasattr(response, 'usage'):
            total_tokens = response.usage.total_tokens
            prompt_tokens = response.usage.prompt_tokens
            completion_tokens = response.usage.completion_tokens
            print(f"    Tokens - Prompt: {prompt_tokens}, Completion: {completion_tokens}, Total: {total_tokens}")
        
        # Small delay to avoid rate limits
        import time
        time.sleep(1)
    
    print("\n" + "=" * 70)
    print(f"FINAL TOTAL TOKENS FOR FULL CONVERSATION: {total_tokens}")
    print("=" * 70)
    print("Each time we pay for ALL previous messages in the conversation history!")

demonstrate_token_growth()

TOKEN GROWTH DEMONSTRATION

[1] User: Hi, I'm Jay!
[1] Assistant: Hi Jay! 👋 Nice to meet you. How can I help you today?
    Tokens - Prompt: 84, Completion: 43, Total: 127

[2] User: I live in surat
[2] Assistant: Great! Surat is a bustling city with a lot to offer. Is there anything specific you’d like to know or talk about—like local events, food recommendations, travel tips, or something else? Let me know how I can help!
    Tokens - Prompt: 115, Completion: 100, Total: 215

[3] User: I work as an computer engineer working in aiml
[3] Assistant: That’s awesome, Jay! Working as a computer engineer in the AI/ML space opens up a lot of exciting possibilities. 

Here are a few ways I can help you right now:

| Area | How I can assist |
|------|-------------------|
| **Project ideas & design** | Brainstorm novel AI/ML projects, outline architectures, suggest datasets, evaluation metrics, or deployment strategies. |
| **Learning & upskilling** | Curated learning paths, recommended books, 